# LAB 3: Electrocardiograms and Neural Networks

In this lab, we will predict heart arrhythmia with LSTMs and 1-D CNNs.

We will use MIT-BIH Arrythmia dataset (https://www.physionet.org/content/mitdb/1.0.0/).
It consists of ECG recordings of several patients with sample rate 360 per second. Experts annotated/classified specific points in the signals as normal, abnormal, or non beat.

**NOTE 1:** GPU is required for this lab.
You must change "Runtime type" to GPU from the "Runtime" tab ("Change Runtime type").

**NOTE 2:** If you face RAM/memory overflow issue in this lab, ensure that you train only one model in single COLAB session. Feel free to implement your own memory optimization tricks too.

**NOTE 3** Google currently offers free Colab Pro for students for one year:
https://blog.google/products-and-platforms/products/education/colab-higher-education/
I recommend signing up for it. It will make your life much easier and save you a lot of waiting time.

In [3]:
# Install the package you need for this lab
# You may need to install them every time you restart the runtime
%pip install wfdb

In [7]:
# Feel free to add more libraries if you need them
import os
import wfdb
import numpy as np
import pandas as pd
import urllib.request
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy.signal

In [8]:
# Load database, this will take a few minutes

path_dataset = 'mba_dataset/mit-bih-arrhythmia-database-1.0.0/'

# Check if the data folder exists
if not os.path.exists(path_dataset):
    print("Dataset not found. Downloading and unzipping...")

    # Download the zip file
    url = 'https://physionet.org/static/published-projects/mitdb/mit-bih-arrhythmia-database-1.0.0.zip'
    urllib.request.urlretrieve(url, './mba_dataset.zip')

    # Unzip the contents
    !unzip -q mba_dataset.zip -d mba_dataset
else:
    print("Dataset already exists.")

Dataset already exists.


## Explore database

We will primarily use \<patientID>.atr files (patientID is being 100, 101, etc.). There are 48 patients with a 30 minutes of recording for each.
For patient '200', let us check what annotations are present in his/her signal. Run the code below.

In [23]:
# We provide function for loading an ECG file
def load_ECG_file(path):
    '''
    Input: path for patient files (excluding extension)
    Output: ECG signal, symbols (labels), indices for such symbols
    '''
    record = wfdb.rdrecord(path)
    annotation = wfdb.rdann(path, 'atr')
    signal = record.p_signal[:,0]    # ECG signal
    symbol = annotation.symbol  # symbols
    index = annotation.sample  # annotation index
    return signal, symbol, index
signal, symbol, index = load_ECG_file(os.path.join(path_dataset, '200'))  # check for yourself if "signal" is ~30 min duration
print(f'Length of signal: {signal.shape}.\n{type(signal)=}\n')  # ECG signal length (in terms of samples and NOT seconds); think why this corresponds to ~30 minutes of signal
print(f'Annotation symbols: {symbol}.\n{type(symbol)=}\n')  # annotation symbols
print(f'Annotation indices: {index}.\n{type(index)=}\n')  # annotion indices of symbols for patient

Length of signal: (650000,).
type(signal)=<class 'numpy.ndarray'>

Annotation symbols: ['+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'V', 'N', 'V', 'N', 'V', 'V', 'N', 'V', 'N', 'V', 'V', 'N', 'V', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'V', 'N', 'V', 'V', 'N', 'V', 'N', 'N', '+', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', '+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', '+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'V', 'N', 'V', 'V', 'N', '+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', 'N', '+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', '+', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', 'N', '+', 'V', 'V', 'V', '+', 'N', 'V', 'N', 'V', 'N', 'V', 'N', 'V', '+', 'N', 'N

We can see a lot of 'N' symbols. It refers to "normal" heartbeat.
The list of abnormal symbols are provided below, which we will consider as "abnormal" beats in this lab.
Any other symbol encountered in the dataset refer to "non-beat" and you **must** skip them in data preparation.
For detailed explanation about symbols, refer to https://archive.physionet.org/physiobank/annotations.shtml

In [26]:
# list of symbols for abnormal beats
abnormal = ['L','R','V','/','A','f','F','j','a','E','J','e','S']

# NOTE: Create an abnoraml set for faster access time
abnormal_set = set(abnormal)

## TASK 1 Data visualization (10 points)
---
Plot any random 10-second long portion of this ECG file (patient 200). Then plot any 1-second portion of this ECG file (patient 200) which has an abnormality approximately in the middle of the signal.

1. How many samples are in a 10-second window?

    Answer:

2. If you want a 1-second window centered at an R-peak index
r, what slice indices would you use (in samples) to get the correct length? Express this as a Python slice: `signal[start:end]`

    Answer:

In [ ]:
# Plot 1
# Your code here

In [ ]:
# Plot 2
# Your code here

##TASK 2 Data preparation (10 points):
---
Training data or test data is usually represented by a matrix $X \in \mathbb{R}^{N\times D}$. N represents the number of training points, and D represents the data dimension. We will consider one data point as +/- 0.5 seconds sequence of samples centered around a normal or abnormal symbol. Therefore, $D = f$ , where $f$ is the sample rate. Your goal is to construct such data matrix $X$. Your function should also output the corresponding label vector $y \in \mathbb{R}^{N\times 1}$. Labels should be 0 for Normal and 1 for abnormal. You should get close to a total of 100k data points.

**Note 1:** Don't forget to ingore the non-beat symbols.

**Note 2:** Labels in abnormal list should all be set with label 1.

**Note 3:** The patients in the training set should be di
fferent from the patients in the test set. We have split the patients for you and you can simply use the patient IDs below.

**Note 4** There might be cases where the 1 second segment also contains other beats. For the simplicity of this lab we can just assume each segment contains exactly one
beats


Why is 𝐷=𝑓? You can either explain with plain language or prove with simple math.


Answer:

In [ ]:
# training patient IDs
pts_train = ['100','101','102','103','104','105','106','107',
       '108','109','111','112','113','114','115','116',
       '117','118','119','121','122','123','124','200',
       '201','202','203','205','207','208','209','210']

# testing patient IDs
pts_test = ['212','213','214','215','217','219','220','221',
       '222','223','228','230','231','232','233','234']

In [ ]:
def make_dataset(pts, num_sec, fs, abnormal):
    '''
    Create a dataset from ECG signals by extracting specific segments around heartbeats
    and labeling them based on the presence of abnormalities.

    Parameters:
    - pts: List of patient identifiers to load ECG files for processing.
    - num_sec: Number of seconds to include before and after each heartbeat in the extracted segment.
    - fs: Sampling frequency of the ECG signals, used to calculate the number of samples in each segment.
    - abnormal: List of abbreviations for abnormalities to look for in the ECG signals.

    Returns:
    - X_all (numpy.ndarray): signal (nbeats , 2 * num_sec * fs columns)
    - Y_all (numpy.ndarray): binary is abnormal (nbeats, 1)
    '''

    pass


def build_XY(p_signal, df_ann, num_cols, ab_idx, abnormal):
    '''
    Build the X (signal segments) and Y (abnormality labels) matrices for each beat.

    Parameters:
    - p_signal: The full ECG signal array.
    - df_ann: DataFrame containing filtered heartbeat annotations.
    - num_cols: The total number of samples per extracted segment.
    - ab_idx: Numpy array containing indices of abnormal beats.
    - abnormal: List of abnormal beat abbreviations.
    - num_sec: Number of seconds before and after each heartbeat for segment extraction.
    - fs: Sampling frequency (Hz).

    Returns:
    - X (numpy.ndarray): Extracted heartbeat signal segments.
    - Y (numpy.ndarray): Binary labels indicating abnormality presence.
    '''

    pass


num_sec =
fs =
X_train, y_train = make_dataset(pts_train, num_sec, fs, abnormal)
X_test, y_test = make_dataset(pts_test, num_sec, fs, abnormal)

In [ ]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
np.sum(y_train,axis=1).sum()/70384

## Task 3 Feature extraction (10 points)
---
You will use Signal Processing library [scipy.signal](https://docs.scipy.org/doc/scipy/reference/signal.html) to extract features for training and testing data matrices. We will do consecutive Fourier transforms to extract the spectrogram of ECG signal.

**Note 1:** A common format of spectrogram is a graph with two geometric dimensions: one axis represents time, and the other axis represents frequency; a third dimension indicating the amplitude of a particular frequency at a particular time is represented by the intensity or color of each point in the image.

Examples of ECG spectrogram are shown below:

![ECG_spetrogram.jpg](https://d3i71xaburhd42.cloudfront.net/c86f76b6eeaa1ae92e5c96a68ca47d046fc01b2e/3-Figure2-1.png)

M. Salem, S. Taheri and J. Yuan, "ECG Arrhythmia Classification Using Transfer Learning from 2- Dimensional Deep CNN Features," 2018 IEEE Biomedical Circuits and Systems Conference (BioCAS), Cleveland, OH, USA, 2018, pp. 1-4, doi: 10.1109/BIOCAS.2018.8584808.

In [ ]:
def extract_features(X):
  '''
  Input: X (N x D): Input data matrix
  Output: F (N x Time x Frequency): Feature matrix
  '''
  pass

F_train = extract_features(X_train)
F_test = extract_features(X_test)
print(F_train.shape, F_test.shape)

## Task 4 Distribution and compensation for imbalance (10 points)
---
Plot the distribution of percentage of oberservations with normal and abnormal labels in training set. Is the dataset balanced? If not, how could you compensate it during training? Would it be helpful?

Answer:

In [ ]:
# Train
# Your code here

In [ ]:
# Test
# Your code here

Write your own BCE loss function and explain briefly how it can help when you have an imbalanced data set.

Answer:

In [ ]:
def mybce(yhat, y, pos_weight, eps=1e-7):
    """
    Custom Binary Cross-Entropy (BCE) loss function with numerical stability and adjustable positive class weight.

    Parameters:
    - yhat (Tensor): Predicted probabilities (should be in the range [0, 1])
    - y (Tensor): Ground truth labels (0 or 1)
    - pos_weight (float): Weight for the positive class to handle class imbalance
    - eps (float): Small constant to prevent log(0) errors

    Returns:
    - loss (Tensor): Computed BCE loss
    """
    pass

**Note:** Be careful when you create your own BCE weighted loss functions, you are required to use this loss instead of nn.BCEWithLogitsLoss in your following task.

# Implementing Neural Networks

In the tasks below you will implement various neural networks you have seen in the lectures. You will train your neural network with the ECG data you have explored above and report the performance of your models.

**Note:** You will be asked to use only torch.nn.Sequential in Task 6 and only torch.nn.Module in Task 7. Points will be deducted if you fail to follow the instructions even if your model is correct

## TASK 5 LSTM (35 points total)
---
 Now, we are going to train a classifier to detect abnormal ECG cycles. We will train a simplified version of the LSTM-based network described in one of the [previously cited papers](https://www.sciencedirect.com/science/article/pii/S0010482518300738?casa_token=qrJ6hAf9tkYAAAAA:7uXqrKY5WqUM6Mjc_qg7wJ4R6QA02BGFXP0o_pOKN09yB8JIXb7067JZWY88rZc8M1G6gkkA).


**Note 1:** Printing loss function and accuracy while training to make sure your model works.

**Note 2:** You need to add a flattening layer after LSTM layer (and before linear layer).

**Note 3:** The output of LSTM in pytorch lib have a tuple outout, add the following GetLSTMOutput after your layer

**Note 4:** You should use the BCE loss fcuntion you implemented above

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score,f1_score

In [ ]:
# Adapt your input to tensor
F_train = torch.from_numpy(F_train).float()
F_test= torch.from_numpy(F_test).float()
y_train = torch.from_numpy(y_train).float()
y_test = torch.from_numpy(y_test).float()

In [ ]:
class GetLSTMOutput(nn.Module):
    def forward(self, x):
        out,_ = x
        return out

### Task 5.1 (8 points)
 Using Pytorch, create a single layer Bidirectional LSTM model using torch.nn.Sequential (i.e. you should NOT use torch.nn.Module). Followed by LSTM layer, you should have linear layer with sigmoid activation and number of output units equals to 1.

   **Note:** You should not use torch.nn.Module. Points will be deducted if you do even if the function is correct

In [ ]:
# Create TensorDatasets
# Your code here

# Create DataLoaders
# Your code here

In [ ]:
# Define hyperparameters
# Your code here

# Create the BiLSTM model using nn.Sequential
# Your code here

### Task 5.2 (2 points)

Define your model, loss function and optimizer. The loss function should be the one you implemented above in Task 5.

In [ ]:
# Define model, loss function and optimizer.
# Your code here


### Task 5.3 (10 points)

Train and test your model.

Plot loss and accuracy with respect to training epochs.

Report your accuracy on test set and training set.

You are required to use the dataloaders you created earlier for mini-batcing

You need to train the model for at least 20 epochs

In [ ]:
# Train the model
# Your code here

# HACK: 
for i in range(100): 
    # 

In [ ]:
# Evaluate the model on the test data, print test accuracy and F1-score
# Your code here

### Task 5.4 (15 points): Pooling for `model_LSTM` (mean vs max)

In Task 5.1, you implemented `model_LSTM` using `nn.Sequential`. The LSTM returns a **sequence** of outputs over time, so before the final `nn.Linear(...)` you must convert that sequence into a fixed-length vector for each sample.

In Task 5.1, the LSTM output has shape `(B, T, F)` (batch, time, features). A linear layer expects a 2D tensor `(B, D)`, so `nn.Flatten()` was used to reshape:
- from `(B, T, F)` to `(B, T*F)`

This keeps **all time steps** by concatenating them into one long vector.

#### How pooling is different from flattening
Pooling also converts `(B, T, F)` into a 2D tensor, but it does so by **aggregating over the time dimension**:
- Flatten: `(B, T, F) -> (B, T*F)` (concatenate time steps)
- Pooling: `(B, T, F) -> (B, F)` (summarize time)

In this task, you will replace `nn.Flatten()` with one of two pooling operations:

1) **Mean pooling** over time: average across the `T` dimension  
2) **Max pooling** over time: elementwise maximum across the `T` dimension

#### What you need to do
Create **two versions** of `model_LSTM` that differ only in the pooling layer:
- `model_LSTM` with **mean pooling**
- `model_LSTM` with **max pooling**

Keep everything else identical to Task 5.3:
- You must use `torch.nn.Sequential`
- same `train_loader` / `test_loader` (mini-batching),
- same loss function `mybce`,
- same optimizer setup (Adam + same learning rate),
- same number of epochs,
- threshold 0.5 for converting probabilities to predicted labels.

#### What to report
Print exactly two lines using the format below:

```
Pooling=mean, Train Loss=..., Train Acc=...%, Test Acc=...%, Test F1=...
```
```
Pooling=max, Train Loss=..., Train Acc=...%, Test Acc=...%, Test F1=...
```



In [ ]:
# Your code here

Compare BiLSTM performance with **no pooling** (`nn.Flatten()`), **mean pooling**, and **max pooling**. Which is best on test F1? Does pooling help? Briefly explain why pooling improved or hurt performance in your case.

Answer:

## TASK 6 1-D CNNs (25 points total)
---
In addition to a LSTM model, we will use [1D CNN](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) with ReLU activation. You need to add a flattening layer just after this (and before linear layer).



**Note 1:** Printing loss function and accuracy while training to make sure your model works.

**Note 2:** You need to add a flattening layer after CNN layer (and before linear layer).

**Note 3:** Sigmoid is recommended as the activation of your last linear layer.

**Note 4:** You should use the loss fcuntion you implemented above

### Task 6.1 (10 points)

 Using Pytorch, create a deep CNN model (more than 1 layer) subclassing torch.nn.Module (i.e. you should NOT use torch.nn.Sequential). Followed by CNN layer, you should have one or several linear layers with different kinds of activation and number of final output units equals to 1.

  **Note:** You should not use torch.nn.Sequential. Points will be deducted if you do even if the function is correct

In [ ]:
# Define hyperparameters
# Your code here

# Create the CNN model using nn.Module
# Your code here

### Task 6.2 (2 points)

 While creating your deep CNN model, how do you validate your model and optimize the hyperparameters (hidden size, learning rate) and optimizer for your model? (Design your method and show how it works in this task)

In [ ]:
# Define model, loss function and optimizer

# Your code here

### Task 6.3 (10 points)

Train and test your model.

Plot loss and accuracy with respect to training epochs.

Report your accuracy on test set and train set.

You need to train the model for at least 20 epochs

##Task 7 Alexnet (125 points total)
---
 AlexNet is a deep convolutional neural network (CNN) designed by Alex Krizhevsky, Ilya Sutskever, and Geoffrey Hinton, which achieved a significant breakthrough in the field of computer vision by winning the ImageNet Large Scale Visual Recognition Challenge (ILSVRC) in 2012. It was one of the first deep neural networks to use multiple layers and dropout regularization to prevent overfitting.

**Note 1**: You can use linear layers to represent FC layers which are shown in the figure.

**Note 2**: You can add dropout layer if needed.

**Note 3**: You can set hyperparameters yourself.

**Note 4**: You may use the built-in `nn.BCEWithLogitsLoss` for this task.

In [ ]:
# Your code here

### Task 6.4 (3 points)

 Comment the difference between your CNN and LSTM models. Which one is better? And why?

Answer:

### Task 7.1 (10 points)

 Establish and train a AlexNet which is similiar to the AlexNet of the paper: ["Classification of ECG signal using FFT based improved Alexnet classifier"](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9514660/) The structure of Alex Net is shown below.

![AlexNet](https://www.ncbi.nlm.nih.gov/corecgi/tileshop/tileshop.fcgi?p=PMC3&id=166618&s=150&r=1&c=1)

In [ ]:
# Create the AlexNet model using either nn.Sequential or nn.Module
# Your code here

# Define loss function and optimizer
# Your code here

### Task 7.2 (10 points)

Train and test your model.

Plot loss and accuracy with respect to training epochs.

Report your accuracy on test set and train set.

You need to train the model for at least 20 epochs

In [ ]:
# Your code here

### Task 7.3 (15 points): Early stopping

In Task 7.2 you trained AlexNet for a fixed number of epochs. In this task, you will implement **early stopping**.

1) Create a **validation set** by splitting the training set into **train/val** (suggested: 90% train, 10% val,  do this by spliting patient ID).  
2) Train for **50 epochs maximum**, but stop early if the **validation loss** has not improved for **patience = 5** consecutive epochs.  
3) Report the epoch where training stopped and the best validation loss achieved.
4) After training stops, restore the model weights from the best validation epoch (the saved best state) and evaluate the test set using that restored model. Report test Accuracy and F1 from the best state (not the final epoch).

**Output requirement:**
1) print one line per epoch in this format:

`Epoch [e/50], Train Loss: ..., Val Loss: ...`

2) At the end print:

`Early stopping at epoch E. Best epoch was B with Val Loss: ...`

3) Report test set accuracy and F1

In [ ]:
# Create a validation set by splitting the training set into train/val
# Your code here

In [ ]:
# Print the following. You should print patient IDs.
print("Train pts:", patients_train)
print("Val pts:", patients_val)

In [ ]:
# Extract features, rebuild TensorDatasets and Dataloaders with the train/val/test splits
# Your code here

In [ ]:
# Define loss function and optimizer
# Your code here

In [ ]:
# Train and test your model

Explain why early stopping can help in general. Did it help in your case?

Answer:

## Bonus Task (15 points)
---

Rebuild the training and test datasets using a new observation definition.

### Part A: Construct X and y
Each beat in MIT-BIH has an annotation (Normal or Abnormal). Define one observation as the ECG segment containing five consecutive beats:
- choose a center beat with a label
- include the two beats before and two beats after
- extract the signal from the sample index of the first beat to the fifth beat

Create:
- X = {x_i} for i=1..N, where each x_i is a 1D signal segment with length D_i (variable across samples)
- y is an N-by-1 label vector, where each region (heart beat) has its own label (0 = Normal, 1 = Abnormal)

Write a detailed explanation of how the five beats are selected (indexing logic, boundary cases) and how segment start/end are defined.


### Part B: Repeat Tasks
Repeat Task 3, Task 4, and Task 5, using the new dataset. In this case, the LSTM will have to provide a sequence as an output, labeling each heart beat, instead of a single pooled output.

### Required plots
Provide plots showing correctness: Plot at least 2 segments from the test set that include Normal and Abnormal heartbeats  with the five beat locations marked and the prediction of the LSTM for each of them. Compare them with the ground truth.


Your colab notebook link:



**You are ready to submit in Gradescope!**

Please suffix your colab file with _\<JHED ID\> (It's the part before the @ symbol in your email)

e.g. Lab3_ECG_LSTM_CNN_myjhedID

4 easy steps to submit your lab:

1.   Click on "Share" option on top right - Click on "copy link" option. Make sure your permission is set to "Anyone on the internet with this link can view". And paste it in the cell above.
2.   Go to "File" - "Download .ipynb" and "Download .py".
3.   Export the notebook to a PDF file with all the outputs.
3.   Upload the three files (.pdf, .ipynb, .py) to Gradescope.

That's it!